<a href="https://colab.research.google.com/github/Existanze54/sirius-machine-learning-2026/blob/main/Seminars/S4_LogReg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 4. Логистическая регрессия

### Давайте импортируем все разом

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

Для удобства можно написать подобную функцию

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score

def train_test_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
        if len(model.classes_) == 2:
            auroc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auroc = roc_auc_score(y_test, y_proba, multi_class="ovr")
    else:
        auroc = None

    acc = accuracy_score(y_test, y_pred)
    return pd.DataFrame({'Acc': [acc], 'AUROC': [auroc]})

### Задача 1. Предсказание вероятностей

#### Подготовка данных


Давайте сгенерируем простой учебный датасет для задачи классификации.

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(
    n_samples=200,
    centers=[[-1.5, 0], [1.5, 0]],
    cluster_std=1.0,
    random_state=0,
)

plt.scatter(X[y==0, 0], X[y==0, 1], label='Class 0')
plt.scatter(X[y==1, 0], X[y==1, 1], label='Class 1')
plt.legend()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

#### Подсмотрим как справляется классификатор Ridge

Почему мы не можем решать задачу классификации методом наименьших квадратов? А мы можем!

In [ ]:
model = RidgeClassifier()
train_test_model(model, X_train, X_test, y_train, y_test)

In [ ]:
model.fit(X_train, y_train)
y_hat = model.decision_function(X_train)

plt.hist(y_hat, bins=25)
plt.show()

Создадим массив с перебором всех значений одной из фичи. Рассчитаем вероятности классов в нескольких бинах и предсказываемый скор.

In [ ]:
x0 = np.arange(-4, 4.1, 0.1)
X_dummy = np.column_stack([x0, np.zeros(x0.shape)])
pd.DataFrame(X_dummy).head(3)

In [ ]:
def calc_proba_by_bins(x, y, n_bins=10):
    bins = np.linspace(x.min(), x.max(), n_bins)
    idx = np.digitize(x, bins)
    proba = np.array([y[idx == i].mean() if np.any(idx == i) else np.nan
                      for i in range(1, len(bins))])
    x_mid = (bins[:-1] + bins[1:]) / 2
    return x_mid, proba

In [ ]:
from sklearn.preprocessing import minmax_scale

y_hat = model.decision_function(X_dummy)
plt.plot(x0, minmax_scale(y_hat))

x_mid, proba = calc_proba_by_bins(X[:, 0], y)

plt.scatter(x_mid, proba)
plt.legend(['rdige proba', 'real proba'])
plt.show()

#### Логистическая регрессия

Обучите логистическую регрессию. Оцените ее точность. Постройте график предсказания вероятностей для сравнения с моделью выше.

In [ ]:
# your code here

### Задача 2. Многоклассовый случай

Давайте научимся классифицировать цветочки. Довольно известен датасет из трех видов ирисов, собранный Рональдом Фишером. Три класса хорошо линейно разделяются и подходят для учебных целей.

In [ ]:
from sklearn.datasets import load_iris
data = load_iris()
X, y = data.data, data.target

In [ ]:
plt.scatter(X[:, 0], X[:, 2], c=y)
plt.show()

Разделите выборку и стандартизуйте признаки. Обучите `LogisticRegression`, `LogisticRegressionCV` со значениями `ovr` и `multinomial` параметра `multi_class`. Постройке столбчатую диаграмму для сравнения качества по метрике $ROC AUC$.

In [ ]:
# your code here